# End-to-End Species Classification With Palmer Penguins

**Purpose:** build a complete public-CSV machine-learning workflow: data ingestion, schema checks, cleaning, preprocessing, model comparison, final holdout evaluation, repeated cross-validation, interpretation, feature-ablation, and error review.

**Dataset:** Palmer Penguins CSV from the `palmerpenguins` project by Allison Horst, Alison Hill, and Kristen Gorman. The project is published under a CC0 license and is commonly used as a modern alternative to Iris.

**Method:** load the public CSV from GitHub, validate expected columns, split once into train/test, compare candidate classifiers with preprocessing inside each pipeline, summarize the selected pipeline with repeated stratified cross-validation on training data, then evaluate it once on the held-out test split.

**Metric:** macro F1 for model selection and final classification summary, because all species should matter rather than optimizing only the majority class.

**Headline takeaway:** this notebook is a compact end-to-end public-CSV workflow. Repeated training-only cross-validation shows metric spread, while the untouched holdout remains the primary result. The feature-ablation section checks whether performance depends on collection-context shortcuts.


## Imports And Project Helpers

The notebook imports the reusable evaluation helper from `src/ml_portfolio`, which keeps repeated metric logic out of the notebook body.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_portfolio import classification_summary
from ml_portfolio.evaluation import repeated_classification_summary
from ml_portfolio.penguins import DATA_URL, load_penguin_data, validate_penguin_columns
from ml_portfolio.plotting import ACCENT, CAPTION, HIGHLIGHT, MUTED, apply_portfolio_style, save_figure

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "assets").exists() and (PROJECT_ROOT.parent / "assets").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

apply_portfolio_style()

RANDOM_STATE = 42



## Load And Validate The Public CSV

The notebook fails fast if the public CSV schema changes. The URL is pinned to a source-repository commit rather than a moving branch so the public-data dependency is reproducible in CI.

In [ ]:
penguins = load_penguin_data(DATA_URL)
validate_penguin_columns(penguins)

print(f"Raw rows: {len(penguins)}")
print(f"Columns: {list(penguins.columns)}")

## Audit Missingness And Target Balance

The modeling version drops rows with missing feature values for a clean teaching benchmark. The pipeline still includes imputers so the same preprocessing contract can handle future inference rows.


In [ ]:
features = [
    "island",
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "sex",
    "year",
]
target = "species"

missingness = penguins[[target, *features]].isna().sum().rename("missing_count")
display(missingness.to_frame())

modeling_data = penguins.dropna(subset=[target, *features]).copy()
modeling_data["year"] = modeling_data["year"].astype(str)

print(f"Rows after dropping missing modeling values: {len(modeling_data)}")
display(modeling_data[target].value_counts().rename("count").to_frame())


## Create A Stratified Holdout Split

Species are separated before any model comparison. The holdout split is used only after candidate selection.


In [ ]:
X = modeling_data[features]
y = modeling_data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")


## Define The Preprocessing Contract

Numeric fields are median-imputed and scaled. Categorical fields are most-frequent imputed and one-hot encoded inside the same pipeline as the classifier.


In [ ]:
numeric_features = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
]
categorical_features = ["island", "sex", "year"]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocess = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features),
    ]
)


## Define Candidate Models

All candidates use the same preprocessing contract, so cross-validation compares model behavior rather than inconsistent feature handling.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=2,
        random_state=RANDOM_STATE,
        n_jobs=1,
    ),
    "kNN": KNeighborsClassifier(n_neighbors=7),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## Compare Candidates On Training Folds

Macro F1 gives each species equal weight during model selection.


In [ ]:
rows = []
for model_name, classifier in models.items():
    pipeline = Pipeline(steps=[("preprocess", preprocess), ("classifier", classifier)])
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="f1_macro")
    rows.append(
        {
            "model": model_name,
            "cv_macro_f1_mean": scores.mean(),
            "cv_macro_f1_std": scores.std(),
        }
    )

cv_results = pd.DataFrame(rows).sort_values(
    ["cv_macro_f1_mean", "cv_macro_f1_std", "model"],
    ascending=[False, True, True],
)
display(cv_results)


## Select The Final Pipeline

The selected estimator is rebuilt as a full preprocessing-plus-model pipeline before holdout evaluation.


In [ ]:
selected_name = str(cv_results.iloc[0]["model"])
selected_model = Pipeline(
    steps=[("preprocess", preprocess), ("classifier", models[selected_name])]
)
print(f"Selected model by cross-validated macro F1: {selected_name}")


## Final Holdout Evaluation

The selected pipeline is fit on the full training split and evaluated once on the test split.


In [ ]:
selected_model.fit(X_train, y_train)
y_pred = selected_model.predict(X_test)
summary = classification_summary(y_test, y_pred, model_name=selected_name)
summary_table = pd.DataFrame([summary])
display(summary_table)


## Repeated Training-Only Cross-Validation

The selected pipeline is checked with five folds repeated three times on the training split only. The mean and standard deviation show how much the educational benchmark moves across stratified resamples; they do not replace the untouched holdout result.


In [ ]:
repeated_cv = repeated_classification_summary(
    selected_model,
    X_train,
    y_train,
    n_splits=5,
    n_repeats=3,
    random_state=RANDOM_STATE,
)
display(pd.DataFrame([repeated_cv]))


## Confusion Matrix

The matrix shows whether the strong headline metric hides any species-specific failure.


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    cmap="Blues",
    values_format="d",
)
plt.title(f"Palmer Penguins holdout confusion matrix: {selected_name}")
plt.tight_layout()
plt.show()


## Interpret Feature Influence

Permutation importance is computed against the fitted pipeline using the raw test-frame columns. This keeps the interpretation at the original feature level.


In [ ]:
importance = permutation_importance(
    selected_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=20,
    random_state=RANDOM_STATE,
)
importance_table = pd.DataFrame(
    {
        "feature": X_test.columns,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }
).sort_values("importance_mean", ascending=False)
display(importance_table)


## Plot Permutation Importance

The plot is useful for GitHub review, but the table above is the source of the interpretation.


In [ ]:
plot_importance = importance_table.sort_values("importance_mean")
fig, ax = plt.subplots()
colors = [HIGHLIGHT if feature == plot_importance["feature"].iloc[-1] else ACCENT for feature in plot_importance["feature"]]
bars = ax.barh(plot_importance["feature"], plot_importance["importance_mean"], color=colors)
ax.bar_label(bars, labels=[f"{value:.3f}" for value in plot_importance["importance_mean"]], padding=3, fontsize=9)
ax.set_xlabel("Mean decrease in macro F1 after permutation")
ax.set_ylabel("Feature")
ax.set_title("Bill measurements carry most of the Penguins signal")
ax.text(
    0,
    -0.22,
    "Palmer Penguins holdout split; permutation importance computed on raw input columns.",
    transform=ax.transAxes,
    color=CAPTION,
    fontsize=9,
)
fig.tight_layout()
save_figure(fig, "palmer_penguins_permutation_importance", project_root=PROJECT_ROOT)
plt.show()

## Feature-Ablation Check

`island` and `year` can behave like collection-context shortcuts. This check compares the full feature set with morphology-only and island-only variants on the same holdout split. It is a diagnostic of shortcut sensitivity, not an independent generalization estimate.


In [ ]:
def build_feature_pipeline(numeric_columns, categorical_columns):
    transformers = []
    if numeric_columns:
        transformers.append(("numeric", numeric_transformer, numeric_columns))
    if categorical_columns:
        transformers.append(("categorical", categorical_transformer, categorical_columns))
    return ColumnTransformer(transformers=transformers)


def evaluate_feature_set(numeric_columns, categorical_columns, label):
    columns = numeric_columns + categorical_columns
    pipeline = Pipeline(
        steps=[
            ("preprocess", build_feature_pipeline(numeric_columns, categorical_columns)),
            ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
        ]
    )
    pipeline.fit(X_train[columns], y_train)
    predictions = pipeline.predict(X_test[columns])
    return classification_summary(y_test, predictions, model_name=label)

ablation_results = pd.DataFrame(
    [
        evaluate_feature_set(numeric_features, categorical_features, "Full feature set"),
        evaluate_feature_set(numeric_features, [], "Morphology only"),
        evaluate_feature_set([], ["island"], "Island only"),
    ]
)
display(ablation_results)


## Error Review

The final check inspects mistakes rather than only reporting a score.


In [ ]:
errors = X_test.copy()
errors["actual_species"] = y_test.to_numpy()
errors["predicted_species"] = y_pred
errors = errors[errors["actual_species"] != errors["predicted_species"]]

print(f"Misclassified holdout rows: {len(errors)}")
if len(errors):
    display(errors)
else:
    print("No misclassified rows in this holdout split.")


## Conclusion

This notebook demonstrates a compact public-CSV workflow: source citation, schema validation, missingness review, typed preprocessing, cross-validated model comparison, repeated training-only uncertainty readout, final holdout evaluation, permutation importance, feature ablation, and error review. The holdout remains the primary result. The ablation makes the collection-context shortcut question visible, but it uses the same holdout and is therefore diagnostic rather than independent validation.
